# Notebook 59 — Physical Parameter Sweep to Universality Phase Diagram

This notebook extends Notebook 58.

Notebook 58 showed the pipeline

`Lindblad-like dynamics → S(x) → Γ_eff(x) → b_local(x) → b`

for a few representative cases.

This notebook scales that up into a **parameter sweep**. We vary simple
effective physical controls and map them into a universality diagram for the
global stretched exponent `b`.

## Main goals

1. define a compact family of Lindblad-like effective-rate models,
2. sweep physical parameters that mimic:
   - dephasing strength,
   - interaction modulation strength,
   - modulation location / width,
3. generate decay curves,
4. fit stretched exponents,
5. visualize a phase map in parameter space.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit


## Grid and helper functions

In [ ]:
x = np.linspace(1e-3, 1.0, 500)

def build_S_from_gamma(gamma_x, x):
    dx = x[1] - x[0]
    integral = np.cumsum(gamma_x) * dx
    return np.exp(-integral), integral

def stretched(x, a, b):
    return np.exp(-a * np.power(x, b))

def fit_stretched(x, S):
    popt, _ = curve_fit(
        stretched,
        np.clip(x, 1e-12, None),
        np.clip(S, 1e-12, 1.0),
        p0=[1.0, 1.0],
        bounds=([0.0, 0.1], [100.0, 3.0]),
        maxfev=50000,
    )
    a, b = [float(v) for v in popt]
    pred = stretched(x, a, b)
    mse = float(np.mean((S - pred) ** 2))
    return a, b, pred, mse

def extract_gamma(S, x):
    return -np.gradient(S, x) / np.clip(S, 1e-12, None)

def b_local_from_gamma(x, gamma_x, S):
    integral = -np.log(np.clip(S, 1e-12, None))
    return x * gamma_x / np.clip(integral, 1e-12, None)


## Lindblad-like effective-rate family

We use a compact rate model with three ingredients:

- baseline decay,
- localized dephasing bump,
- interaction-like sinusoidal modulation.


In [ ]:
def gamma_model(x, gamma0=2.0, deph_amp=0.0, deph_center=0.3, deph_width=0.15,
                int_amp=0.0, int_phase=0.0):
    bump = deph_amp * np.exp(-((x - deph_center) / deph_width)**2)
    interaction = gamma0 * int_amp * np.sin(2 * np.pi * x + int_phase)
    gamma = gamma0 + bump + interaction

    # keep gamma positive and well-behaved
    gamma = np.clip(gamma, 0.2, None)
    return gamma


## Parameter grids

In [ ]:
deph_amps = np.linspace(0.0, 1.0, 11)
int_amps = np.linspace(0.0, 0.45, 11)

gamma0 = 2.0
deph_center = 0.30
deph_width = 0.15
int_phase = 0.0


## Sweep: dephasing vs interaction

In [ ]:
b_map = np.zeros((len(deph_amps), len(int_amps)))
mse_map = np.zeros_like(b_map)

example_rows = []

for i, deph_amp in enumerate(deph_amps):
    for j, int_amp in enumerate(int_amps):
        gamma_x = gamma_model(
            x,
            gamma0=gamma0,
            deph_amp=deph_amp,
            deph_center=deph_center,
            deph_width=deph_width,
            int_amp=int_amp,
            int_phase=int_phase,
        )
        S, I = build_S_from_gamma(gamma_x, x)
        a_fit, b_fit, S_fit, mse = fit_stretched(x, S)

        b_map[i, j] = b_fit
        mse_map[i, j] = mse

        # Save a few representative examples
        if (i, j) in [(0, 0), (5, 0), (0, 7), (8, 8)]:
            example_rows.append({
                "deph_amp": deph_amp,
                "int_amp": int_amp,
                "gamma": gamma_x,
                "S": S,
                "b_fit": b_fit,
                "mse": mse,
            })

print("b range:", float(np.min(b_map)), "to", float(np.max(b_map)))
print("fit mse range:", float(np.min(mse_map)), "to", float(np.max(mse_map)))


## Plot 1 — Example effective-rate processes

In [ ]:
plt.figure(figsize=(8.5, 5.2))
for row in example_rows:
    plt.plot(
        x, row["gamma"],
        label=f'deph={row["deph_amp"]:.2f}, int={row["int_amp"]:.2f}, b={row["b_fit"]:.2f}'
    )
plt.xlabel("x")
plt.ylabel("Γ_eff(x)")
plt.title("Representative effective-rate processes across the sweep")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()


## Plot 2 — Example decay curves

In [ ]:
plt.figure(figsize=(8.5, 5.2))
for row in example_rows:
    plt.plot(
        x, row["S"],
        label=f'deph={row["deph_amp"]:.2f}, int={row["int_amp"]:.2f}, b={row["b_fit"]:.2f}'
    )
plt.xlabel("x")
plt.ylabel("S(x)")
plt.title("Representative decay curves across the sweep")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()


## Plot 3 — Universality phase diagram: fitted exponent b

In [ ]:
plt.figure(figsize=(7.6, 5.8))
im = plt.imshow(
    b_map,
    origin="lower",
    aspect="auto",
    extent=[int_amps[0], int_amps[-1], deph_amps[0], deph_amps[-1]],
)
plt.colorbar(im, label="fitted global exponent b")
plt.xlabel("interaction modulation amplitude")
plt.ylabel("dephasing bump amplitude")
plt.title("Universality phase diagram in physical parameter space")
plt.tight_layout()
plt.show()


## Plot 4 — Fit-quality phase diagram

In [ ]:
plt.figure(figsize=(7.6, 5.8))
im = plt.imshow(
    np.log10(mse_map + 1e-16),
    origin="lower",
    aspect="auto",
    extent=[int_amps[0], int_amps[-1], deph_amps[0], deph_amps[-1]],
)
plt.colorbar(im, label="log10 fit MSE")
plt.xlabel("interaction modulation amplitude")
plt.ylabel("dephasing bump amplitude")
plt.title("Fit quality across the parameter sweep")
plt.tight_layout()
plt.show()


## Cross-sections through the phase diagram

In [ ]:
plt.figure(figsize=(8.2, 5.0))
for deph_amp in [0.0, 0.3, 0.6, 0.9]:
    i = int(np.argmin(np.abs(deph_amps - deph_amp)))
    plt.plot(int_amps, b_map[i, :], marker="o", label=f"deph={deph_amps[i]:.1f}")
plt.xlabel("interaction modulation amplitude")
plt.ylabel("fitted b")
plt.title("Cross-sections: effect of interaction strength")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8.2, 5.0))
for int_amp in [0.0, 0.15, 0.30, 0.45]:
    j = int(np.argmin(np.abs(int_amps - int_amp)))
    plt.plot(deph_amps, b_map[:, j], marker="o", label=f"int={int_amps[j]:.2f}")
plt.xlabel("dephasing bump amplitude")
plt.ylabel("fitted b")
plt.title("Cross-sections: effect of dephasing strength")
plt.legend()
plt.tight_layout()
plt.show()


## Optional second sweep: dephasing position

In [ ]:
deph_centers = np.linspace(0.15, 0.75, 13)
fixed_deph_amp = 0.8
fixed_int_amp = 0.2

b_center_map = np.zeros((len(deph_centers), len(int_amps)))

for i, center in enumerate(deph_centers):
    for j, int_amp in enumerate(int_amps):
        gamma_x = gamma_model(
            x,
            gamma0=gamma0,
            deph_amp=fixed_deph_amp,
            deph_center=center,
            deph_width=deph_width,
            int_amp=int_amp,
            int_phase=int_phase,
        )
        S, I = build_S_from_gamma(gamma_x, x)
        a_fit, b_fit, S_fit, mse = fit_stretched(x, S)
        b_center_map[i, j] = b_fit

plt.figure(figsize=(7.6, 5.8))
im = plt.imshow(
    b_center_map,
    origin="lower",
    aspect="auto",
    extent=[int_amps[0], int_amps[-1], deph_centers[0], deph_centers[-1]],
)
plt.colorbar(im, label="fitted global exponent b")
plt.xlabel("interaction modulation amplitude")
plt.ylabel("dephasing bump center")
plt.title("Shift of universality class with dephasing location")
plt.tight_layout()
plt.show()


## Compact summary

In [ ]:
lines = []
lines.append("Physical parameter sweep to universality phase diagram")
lines.append("")
lines.append(f"b range across sweep: {float(np.min(b_map)):.6f} to {float(np.max(b_map)):.6f}")
lines.append(f"fit MSE range: {float(np.min(mse_map)):.6e} to {float(np.max(mse_map)):.6e}")
lines.append("")
lines.append("Interpretation:")
lines.append("- Physical control parameters map smoothly into universality classes.")
lines.append("- Dephasing-like localized structure and interaction-like modulation shift the fitted exponent.")
lines.append("- The stretched exponent becomes a phase-diagram observable in parameter space.")
print("\n".join(lines))


## Final conclusion

This notebook turns the theory into a **phase-diagram language**.

We show that simple physical control parameters generate structured
effective-rate processes, and those processes map into different values of the
global stretched exponent `b`.

So the final picture is:

physical parameters  
→ `Γ_eff(x)`  
→ `b_local(x)`  
→ global universality exponent `b`

This makes `b` a physically meaningful classification variable, not just a fit parameter.
